# STEP 5 — Analysis & Paper Figures

**Prerequisite:** Steps 2–4 complete.

**Figures produced (all in `figures/`):**
- `fig1_pareto_frontier.png`     — Main result: task success vs LLM call ratio
- `fig2_3level_vs_2level.png`   — Cascade comparison
- `fig3_step_escalation.png`    — Where in trajectory does escalation cluster?
- `fig4_signal_quality.png`     — ROC curves: routing signal AUC comparison
- `fig5_cross_task.png`         — MuSiQue Pareto (generalization check)

**Hypothesis verification printed at the end.**

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path

sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_init import init
P = init()
from src.metrics import compute_summary, compute_pareto_frontier, compute_step_distribution, compute_routing_signal_auc

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'figure.dpi': 150, 'lines.linewidth': 2, 'lines.markersize': 7})

# Load all results
baseline = pd.read_csv(P['results'] / 'baseline_episodes.csv')
routing  = pd.read_csv(P['results'] / 'routing_episodes.csv')
learned  = pd.read_csv(P['results'] / 'learned_router_episodes.csv')
steps_df = pd.read_csv(P['results'] / 'baseline_steps.csv')
all_ep   = pd.concat([baseline, routing, learned], ignore_index=True)

summary = compute_summary(all_ep)
summary = compute_pareto_frontier(summary)
step_dist = compute_step_distribution(steps_df)

print(f'Total episodes: {len(all_ep)}')
print(f'Unique policies: {all_ep["policy"].nunique()}')

In [ ]:
# ── FIGURE 1: Pareto Frontier (HotpotQA) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

ds = 'hotpotqa'
slm = 'Qwen2.5-1.5B'

POLICY_STYLES = {
    'conf_gate':   ('#1f77b4', 'o', 'Conf gate'),
    'entropy_gate':('#d62728', 's', 'Entropy gate'),
    'step_gate':   ('#ff7f0e', '^', 'Step gate'),
    'cascade':     ('#2ca02c', 'D', '3-Level cascade'),
    'learned':     ('#9467bd', 'P', 'Learned router'),
}

for prefix, (color, marker, label) in POLICY_STYLES.items():
    mask = (summary['policy'].str.startswith(prefix)) & (summary['slm_name']==slm) & (summary['dataset']==ds)
    pts  = summary[mask].sort_values('llm_call_ratio')
    if len(pts):
        ax.plot(pts['llm_call_ratio'], pts['task_success'], color=color, marker=marker, label=label, alpha=0.8)

# Anchor points
baseline_rows = summary[summary['dataset']==ds]
for pid, label, m in [('always_slm_1b','Always SLM','x'), ('always_llm','Always LLM (Oracle)','*')]:
    r = baseline_rows[baseline_rows['policy']==pid]
    if len(r):
        ax.scatter(r['llm_call_ratio'], r['task_success'], marker=m, s=150, color='black', zorder=5, label=label)

ax.set_xlabel('LLM Call Ratio (Fraction of steps using LLM)')
ax.set_ylabel('Task Success Rate (Exact Match)')
ax.set_title(f'Figure 1: Pareto Frontier — HotpotQA (SLM: {slm})')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig1_pareto_frontier.png', bbox_inches='tight')
plt.show()
print('✓ fig1 saved')

In [ ]:
# ── FIGURE 2: 3-Level vs Best 2-Level ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ds = 'hotpotqa'

for prefix, color, label in [
    ('entropy_gate', '#d62728', 'Entropy gate (2-level)'),
    ('cascade',      '#2ca02c', '3-Level cascade'),
]:
    mask = (summary['policy'].str.startswith(prefix)) & (summary['dataset']==ds)
    pts  = summary[mask].sort_values('llm_call_ratio')
    ax.plot(pts['llm_call_ratio'], pts['task_success'], color=color, marker='o', label=label)

r_oracle = summary[(summary['policy']=='always_llm') & (summary['dataset']==ds)]
if len(r_oracle):
    ax.axhline(float(r_oracle['task_success'].values[0]), color='gray', linestyle='--', label='Oracle ceiling')

ax.set_xlabel('LLM Call Ratio')
ax.set_ylabel('Task Success Rate')
ax.set_title('Figure 2: 3-Level Cascade vs Best 2-Level Policy')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig2_3level_vs_2level.png', bbox_inches='tight')
plt.show()
print('✓ fig2 saved')

In [ ]:
# ── FIGURE 3: Step Escalation Heatmap ───────────────────────────────────
# Use always_slm_1b baseline steps for entropy vs step_idx
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ds in zip(axes, ['hotpotqa', 'musique']):
    sub = step_dist[(step_dist['dataset']==ds) & (step_dist['slm_name']=='Qwen2.5-1.5B')]
    ax.bar(sub['step_idx'], sub['mean_entropy'], color='#1f77b4', alpha=0.7, label='Mean entropy')
    ax2 = ax.twinx()
    ax2.plot(sub['step_idx'], sub['escalation_rate'] if 'escalation_rate' in sub.columns else [0]*len(sub),
             color='#d62728', marker='o', label='Escalation rate')
    ax.set_xlabel('ReAct Step Index')
    ax.set_ylabel('Mean First-Token Entropy')
    ax2.set_ylabel('Escalation Rate')
    ax.set_title(f'Figure 3: Step-Level Signals — {ds.title()}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(P['figures'] / 'fig3_step_escalation.png', bbox_inches='tight')
plt.show()
print('✓ fig3 saved')

In [ ]:
# ── FIGURE 4: Routing Signal Quality (AUC) ──────────────────────────────
signal_aucs = compute_routing_signal_auc(steps_df)
print('Routing signal AUC:')
print(signal_aucs.to_string(index=False))

if len(signal_aucs):
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['#1f77b4','#d62728','#2ca02c']
    bars = ax.bar(signal_aucs['signal'], signal_aucs['auc'], color=colors[:len(signal_aucs)], alpha=0.8)
    ax.axhline(0.5, color='gray', linestyle='--', label='Random baseline')
    ax.set_ylabel('AUC (predicting SLM failure)')
    ax.set_title('Figure 4: Routing Signal Quality')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(P['figures'] / 'fig4_signal_quality.png', bbox_inches='tight')
    plt.show()
    print('✓ fig4 saved')

In [ ]:
# ── FIGURE 5: Cross-Task Generalization (MuSiQue) ───────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ds = 'musique'

for prefix, (color, marker, label) in POLICY_STYLES.items():
    mask = (summary['policy'].str.startswith(prefix)) & (summary['dataset']==ds)
    pts  = summary[mask].sort_values('llm_call_ratio')
    if len(pts):
        ax.plot(pts['llm_call_ratio'], pts['task_success'], color=color, marker=marker, label=label, alpha=0.8)

ax.set_xlabel('LLM Call Ratio')
ax.set_ylabel('Task Success Rate')
ax.set_title('Figure 5: Cross-Task Generalization — MuSiQue')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(P['figures'] / 'fig5_cross_task.png', bbox_inches='tight')
plt.show()
print('✓ fig5 saved')

In [ ]:
# ── Hypothesis verification ──────────────────────────────────────────────
print('=== HYPOTHESIS VERIFICATION ===\n')

base_hp = summary[(summary['dataset']=='hotpotqa')]
oracle_sr = float(base_hp[base_hp['policy']=='always_llm']['task_success'].values[0]) if len(base_hp[base_hp['policy']=='always_llm']) else 0
slm_sr    = float(base_hp[base_hp['policy']=='always_slm_1b']['task_success'].values[0]) if len(base_hp[base_hp['policy']=='always_slm_1b']) else 0

print(f'H0 — LLM (oracle) SR={oracle_sr:.3f} > SLM SR={slm_sr:.3f}: {oracle_sr > slm_sr}')

# H1: at least one routing policy achieves 90% of oracle SR at < 60% LLM call ratio
routing_pts = base_hp[~base_hp['policy'].isin(['always_slm_1b','always_slm_3b','always_llm'])]
good = routing_pts[(routing_pts['task_success'] >= 0.9 * oracle_sr) & (routing_pts['llm_call_ratio'] < 0.6)]
print(f'H1 — Routing at ≥90% oracle SR with <60% LLM calls: {len(good)>0}')
if len(good):
    best = good.loc[good['llm_call_ratio'].idxmin()]
    print(f'     Best: {best["policy"]} SR={best["task_success"]:.3f} LLM_ratio={best["llm_call_ratio"]:.3f}')

# H2: learned router + entropy gate Pareto-dominate step_gate
pareto_dom = routing_pts[routing_pts['pareto_dominant']]['policy'].unique()
step_dom   = any('step_gate' in p for p in pareto_dom)
ent_dom    = any('entropy_gate' in p for p in pareto_dom)
learn_dom  = any('learned' in p for p in pareto_dom)
print(f'H2 — Uncertainty-based routing on Pareto frontier: entropy={ent_dom}, learned={learn_dom}, step_gate={step_dom}')

print('\n✅ STEP 5 COMPLETE — All figures saved to figures/')